### torch.add

In [2]:
%uv pip install torch

Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 12ms
Note: you may need to restart the kernel to use updated packages.


In [7]:
import torch
import time

In [8]:
def vector_add(x: torch.Tensor, y: torch.Tensor, out: torch.Tensor):
    return torch.add(x, y, out=out)

In [9]:
n = 1 << 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = torch.ones(n, device=device, dtype=torch.float32) # full of 1s
y = torch.full((n,), 2.0, device=device, dtype=torch.float32) # full of 2s
result = torch.empty_like(x)

In [12]:
# warm up cuda before measure
for _ in range(10):
    vector_add(x, y, result)

if device.type == "cuda":
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()
    vector_add(x, y, result)
    end.record()

    torch.cuda.synchronize()
    elapsed_ms = start.elapsed_time(end)
    
else:
    start_time = time.perf_counter()
    vector_add(x, y, result)
    elapsed_ms = (time.perf_counter() - start_time) * 1000

max_error = (result-3.0).abs().max().item()

In [13]:
print(f"Max error: {max_error}, Elapsed time: {elapsed_ms:.3f} ms")

Max error: 0.0, Elapsed time: 0.037 ms
